## Deep Research

One of the classic cross-business Agentic use cases! This is huge.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">A Deep Research agent is broadly applicable to any business area, and to your own day-to-day activities. You can make use of this yourself!
            </span>
        </td>
    </tr>
</table>

In [2]:
from agents import Agent, WebSearchTool, trace, Runner, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
from IPython.display import display, Markdown
from messenger import send_email, push
from agents.extensions.visualization import draw_graph

In [3]:
load_dotenv(override=True)

True

In [4]:
import os
from agents import set_tracing_export_api_key

os.environ["OPENAI_LOG"] = "debug"
# del os.environ["OPENAI_LOG"]
set_tracing_export_api_key(os.getenv("OPENAI_TRACING_KEY"))

In [5]:
# Constants 

MODEL_NAME = "gpt-5.4-mini"
USE_EMAIL = True
HOW_MANY_SEARCHES = 5

## Strategy for the Deep Research Agent

We are going to do it the bulletproof way.

We are going to orchestrate with code: separate calls to `Runner.run()` for each step in the process.

We will use Structured Outputs at each point.

## We will build 4 Agents:

1. The Search Agent: searches the web for information
2. The Planner Agent: given a question, comes up with a list of searches that should be made
3. The Writer Agent: writes a robust report
4. The Emailer Agent: crafts and sends an email

And then 4 python functions, 1 to call Runner.run() for each of the 4 agents.


## Agent 1: The Search Agent

### OpenAI Hosted Tools

https://openai.github.io/openai-agents-python/tools/#hosted-tools

A paid, quick approach to carrying out managed functionality on OpenAI's cloud.

Their docs surface these tools, but it's worth keeping in mind that they're costly and lock you in to the OpenAI ecosystem.

OpenAI offers the following hosted tools:

`WebSearchTool` lets an agent search the web.  
`FileSearchTool` allows retrieving information from your OpenAI Vector Stores.  
`CodeInterpreterTool` lets the LLM execute code in a sandboxed environment.  
`HostedMCPTool` exposes a remote MCP server's tools to the model.  
`ImageGenerationTool` generates images from a prompt.  
`ToolSearchTool` lets the model load deferred tools, namespaces, or hosted MCP servers on demand.  

### Important note - API charge of WebSearchTool

This currently costs 1 cent per call for OpenAI WebSearchTool. That can add up to about $1 for the next 2 labs. We'll use free and low cost Search tools with other platforms, so feel free to skip running this if the cost is a concern. Also student Christian W. pointed out that OpenAI can sometimes charge for multiple searches for a single call, so it could sometimes cost more than 1 cent per call.

Costs are in the Tools section here: https://developers.openai.com/api/docs/pricing


In [6]:
INSTRUCTIONS = """
You are a research assistant. Given a search term, you search the web for that term and 
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 words.
Capture the main points and be succinct. Reply only with the summary.
"""
task = "Most popular AI Agent frameworks in 2026"

require_tools_setting = ModelSettings(tool_choice="required")
tools = [WebSearchTool()]

In [7]:
tools[0]

WebSearchTool(user_location=None, filters=None, search_context_size='medium', external_web_access=None)

In [8]:
search_agent = Agent(name="Search Agent", instructions=INSTRUCTIONS, tools=tools, model=MODEL_NAME, model_settings=require_tools_setting)

In [ ]:
# test the search agent
result = await Runner.run(search_agent, task)
display(Markdown(result.final_output))

### As always, take a look at the trace

https://platform.openai.com/traces

## Agent 2: The Planner Agent

### We will now use Structured Outputs, and include a description of the fields

In [11]:
class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")
    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")

In [12]:
WebSearchPlan.model_json_schema()

{'$defs': {'WebSearchItem': {'properties': {'reason': {'description': 'Your reasoning for why this search is important to the query.',
     'title': 'Reason',
     'type': 'string'},
    'query': {'description': 'The search term to use for the web search.',
     'title': 'Query',
     'type': 'string'}},
   'required': ['reason', 'query'],
   'title': 'WebSearchItem',
   'type': 'object'}},
 'properties': {'searches': {'description': 'A list of web searches to perform to best answer the query.',
   'items': {'$ref': '#/$defs/WebSearchItem'},
   'title': 'Searches',
   'type': 'array'}},
 'required': ['searches'],
 'title': 'WebSearchPlan',
 'type': 'object'}

In [13]:
# See note above about cost of WebSearchTool

INSTRUCTIONS = f"""
You are a research assistant. Given a user query, come up with a set of web searches
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for.
"""

planner_agent = Agent(name="Planner Agent", instructions=INSTRUCTIONS, model=MODEL_NAME, output_type=WebSearchPlan)

In [14]:

result = await Runner.run(planner_agent, task)
result.final_output

WebSearchPlan(searches=[WebSearchItem(reason='Find current market rankings, community adoption, and recent comparisons of AI agent frameworks in 2026.', query='most popular AI agent frameworks 2026 ranking'), WebSearchItem(reason='Identify which frameworks have the largest GitHub activity, downloads, and ecosystem support.', query='AI agent frameworks GitHub stars downloads 2026'), WebSearchItem(reason='Compare leading open-source and commercial agent frameworks with recent reviews and benchmarks.', query='best AI agent frameworks 2026 comparison'), WebSearchItem(reason='Check whether newer frameworks or libraries have overtaken older ones in popularity by 2026.', query='top agentic AI frameworks 2026 new frameworks'), WebSearchItem(reason='Look for surveys, reports, and articles about enterprise adoption of agent frameworks in 2026.', query='enterprise adoption AI agent frameworks 2026 report')])

## Agent 3: The Writer Agent

In [15]:
INSTRUCTIONS = """
You are a senior researcher tasked with writing a cohesive report for a research query.
You will be provided with the original query, and some research.
Generate a comprehensive report based on the research and the query.
The final output should be in markdown format, and it should be lengthy and detailed. Aim 
for 5-10 pages of content, at least 1000 words.
"""


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")
    markdown_report: str = Field(description="The final report")
    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(name="Writer Agent", instructions=INSTRUCTIONS, model=MODEL_NAME, output_type=ReportData)

## Agent 4: The email agent

In [16]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects
    
    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")
    return "Email sent successfully"

In [17]:
send_email_tool.params_json_schema

{'properties': {'subject': {'description': 'The subject of the email',
   'title': 'Subject',
   'type': 'string'},
  'text_body': {'description': 'The body of the email as plain text',
   'title': 'Text Body',
   'type': 'string'},
  'html_body': {'description': 'The HTML body of the email',
   'title': 'Html Body',
   'type': 'string'}},
 'required': ['subject', 'text_body', 'html_body'],
 'title': 'send_email_tool_args',
 'type': 'object',
 'additionalProperties': False}

In [18]:
INSTRUCTIONS = """
You are provided with a detailed report. Use your tool to send an email, converting the report into
a clean, well presented HTML email with an appropriate subject line.
"""

email_agent = Agent(name="Email Agent", instructions=INSTRUCTIONS, tools=[send_email_tool], model=MODEL_NAME)

## Now to Orchestrate by Agents

Steps:
- Gather all the tools in an array by converting the existing agents as tools
- Create the "Research Manager" agent with instructions including tools usage - pass the tools array and force their usage through ModelSettings

In [21]:
search_agent_tool = search_agent.as_tool(tool_name="search_agent", tool_description="Search the web for a given query and return a concise summary of the results.")
planner_agent_tool = planner_agent.as_tool(tool_name="planner_agent", tool_description="Given a user query, come up with a set of web searches to perform to best answer the query.")
writer_agent_tool = writer_agent.as_tool(tool_name="writer_agent", tool_description="Given a user query and some research, generate a comprehensive report based on the research and the query.")
email_agent_tool = email_agent.as_tool(tool_name="email_agent", tool_description="Given a report, send an email with the report as the body of the email.")

In [22]:
TOPIC = "Most popular AI Agent frameworks in 2026"

INSTRUCTIONS = f"""
You are a Researcher who's goal is to research and orchestrate the research process. 

The goal is to research on this topic - {TOPIC}

You have access to the following tools:
- search_agent: Search the web for a given query and return a concise summary of the results.
- planner_agent: Given a user query, come up with a set of web searches to perform to best answer the query.
- writer_agent: Given a user query and some research, generate a comprehensive report based on the research and the query.
- email_agent: Given a report, send an email with the report as the body of the email.

Your task is to use the tools available to you to produce a comprehensive report on the query, and send it via email. 

You should first use the planner_agent to come up with a set of web searches to perform, then use the search_agent to perform those searches and gather research, 
then use the writer_agent to generate a comprehensive report based on the research and the original query, and finally use the email_agent to send the report via email. 

You should only use the tools available to you.
"""
require_tools_setting = ModelSettings(tool_choice="required")

research_manager_agent = Agent(name="Research Manager Agent", instructions=INSTRUCTIONS, tools=[search_agent_tool, planner_agent_tool, writer_agent_tool, email_agent_tool], model=MODEL_NAME, model_settings=require_tools_setting)

In [23]:
draw_graph(research_manager_agent)

ExecutableNotFound: failed to execute PosixPath('dot'), make sure the Graphviz executables are on your systems' PATH

### Showtime!

In [24]:
with trace("Research Manager Agent Execution"):
    result = await Runner.run(research_manager_agent, INSTRUCTIONS)

### As always, take a look at the trace

https://platform.openai.com/traces

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thanks.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00cc00;">Congratulations on your progress, and a request</h2>
            <span style="color:#00cc00;">You've reached an important moment with the course; you've created a valuable Agent using one of the latest Agent frameworks. You've upskilled, and unlocked new commercial possibilities. Take a moment to celebrate your success!<br/><br/>Something I should ask you -- my editor would smack me if I didn't mention this. If you're able to rate the course on Udemy, I'd be seriously grateful: it's the most important way that Udemy decides whether to show the course to others and it makes a massive difference.<br/><br/>And another reminder to <a href="https://www.linkedin.com/in/eddonner/">connect with me on LinkedIn</a> if you wish! If you wanted to post about your progress on the course, please tag me and I'll weigh in to increase your exposure.
            </span>
        </td>
    </tr>